In [ ]:
def visualize_detection_cv(self, screenshot, pacman_pos, ghost_pos, pg_pos):
        """
        Визуализирует обнаруженные объекты на скриншоте.
        """
        vis_img = screenshot.copy()
        # Рисуем пакмана (зеленый круг)
        if pacman_pos[0] is not None and pacman_pos[1] is not None:
            cv2.circle(vis_img, pacman_pos, 15, (0, 255, 0), 3)
            cv2.putText(vis_img, 'PACMAN', (pacman_pos[0]-25, pacman_pos[1]-20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 255, 0), 2)
        # Рисуем призрака (красный круг)
        if ghost_pos[0] is not None and ghost_pos[1] is not None:
            cv2.circle(vis_img, ghost_pos, 15, (0, 0, 255), 3)
            cv2.putText(vis_img, 'GHOST', (ghost_pos[0]-20, ghost_pos[1]-20), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0, 0, 255), 2)
        # Рисуем ближайший пакгам (синий круг)
        if pg_pos[0] is not None and pg_pos[1] is not None:
            cv2.circle(vis_img, pg_pos, 8, (255, 200, 0), 3)
            cv2.putText(vis_img, 'PAC-GUM', (pg_pos[0]-25, pg_pos[1]-15), 
                    cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 200, 0), 2)
        # Добавляем информационную панель
        info_y = 30
        cv2.putText(vis_img, f'Frame: {self.pacman_time}', (10, info_y), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(vis_img, f'Score: {self.game.score}', (10, info_y + 20), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        cv2.putText(vis_img, f'Epsilon: {self.epsilon:.3f}', (10, info_y + 40), 
                cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255, 255, 255), 1)
        # Рисуем линии от пакмана к другим объектам
        if (pacman_pos[0] is not None and pacman_pos[1] is not None and
            ghost_pos[0] is not None and ghost_pos[1] is not None):
            cv2.line(vis_img, pacman_pos, ghost_pos, (0, 100, 255), 2)
        if (pacman_pos[0] is not None and pacman_pos[1] is not None and
            pg_pos[0] is not None and pg_pos[1] is not None):
            cv2.line(vis_img, pacman_pos, pg_pos, (255, 100, 0), 2)
        return vis_img

In [ ]:
def cv_analysis(self):
        """
        Анализ скриншота с помощью компьютерного зрения.
        Определяет позиции объектов и визуализирует результат.
        """
        try:
            # Создаем скриншот текущего экрана
            screenshot = pygame.surfarray.array3d(self.game.screen)
            screenshot = screenshot.transpose([1, 0, 2])  # Преобразуем для OpenCV
            # Конвертируем RGB в BGR (OpenCV использует BGR)
            screenshot_bgr = cv2.cvtColor(screenshot, cv2.COLOR_RGB2BGR)
            # Калибровка по нажатию клавиши 'c'
            keys = pygame.key.get_pressed()
            if keys[pygame.K_c]:
                self.calibrate_colors(screenshot_bgr)
                return
            # Определяем состояние с помощью компьютерного зрения
            cv_state, detected_objects = self.image_to_state_cv(screenshot_bgr)
            
            # Сравнение с внутренним состоянием
            if self.cv_debug:
                internal_state_str = f"Internal: {self.current_state}"
                cv_state_str = f"CV: {cv_state}"
                
                #if self.current_state != cv_state and cv_state != 0:
                #    print(f"STATE MISMATCH! {internal_state_str} vs {cv_state_str}")
                #else:
                #    print(f"States match: {internal_state_str}")
            # Создаем визуализацию
            vis_image = self.visualize_detection_cv(
                screenshot_bgr, 
                detected_objects['pacman'], 
                detected_objects['ghost'], 
                detected_objects['pg']
            )
            # Добавляем информацию о состояниях
            if self.cv_debug:
                cv2.putText(vis_image, f"Internal State: {self.current_state}", 
                        (10, vis_image.shape[0] - 60), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
                cv2.putText(vis_image, f"CV State: {cv_state}", 
                        (10, vis_image.shape[0] - 30), 
                        cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
            # Отображаем визуализацию в отдельном окне
            if self.show_cv_window:
                cv2.imshow('Computer Vision Analysis', vis_image)
                cv2.waitKey(1)  # Короткая задержка для отображения     
        except Exception as e:
            if self.cv_debug:
                print(f"CV Analysis Error: {e}")

In [ ]:
def image_to_state_cv(self, screenshot):
        """
        Определяет состояние из скриншота игры с новыми цветами текстур.
        Улучшенная версия с лучшей детекцией пакмана.
        """
        # ОПРЕДЕЛЯЕМ ГРАНИЦЫ ИГРОВОГО ПОЛЯ
        # Верхняя панель счета: примерно 32 пикселя
        # Нижняя панель с жизнями: примерно 40 пикселей
        top_panel_height = 32
        bottom_panel_height = 40
        # Высота и ширина скриншота
        screen_height, screen_width = screenshot.shape[:2]
        # Вычисляем границы игрового поля
        game_field_top = top_panel_height
        game_field_bottom = screen_height - bottom_panel_height
        game_field_left = 0
        game_field_right = screen_width
        # Обрезаем скриншот до игрового поля
        game_field = screenshot[game_field_top:game_field_bottom, 
                            game_field_left:game_field_right]
        
        if game_field.size == 0:
            # Если не удалось обрезать, используем весь скриншот
            game_field = screenshot
            game_field_top = 0
            game_field_bottom = screen_height
        # Создаем копию для отладки
        debug_img = game_field.copy() if hasattr(self, 'cv_debug') and self.cv_debug else None
        # Более точные диапазоны цветов для пакмана (основной цвет и дополнения)
        # Анализируем предоставленные цвета пакмана: 683b37, 774c47, 422820, e6d1cc, cab2ac, fff5f6, ad918b
        # Основной цвет пакмана: коричневато-розовые оттенки
        pacman_main_colors = [
            # Основные цвета пакмана (телесные/коричневые оттенки)
            np.array([0x37, 0x3b, 0x68]),   # 683b37 -> BGR: 55,59,104 (темный)
            np.array([0x47, 0x4c, 0x77]),   # 774c47 -> BGR: 71,76,119 (средний)
            np.array([0xcc, 0xd1, 0xe6]),   # e6d1cc -> BGR: 204,209,230 (светлый)
            np.array([0xac, 0xb2, 0xca]),   # cab2ac -> BGR: 172,178,202 (средне-светлый)
        ]
        # Дополнительные цвета для лучшего обнаружения
        pacman_secondary_colors = [
            np.array([0xf6, 0xf5, 0xff]),   # fff5f6 -> BGR: 246,245,255 (очень светлый)
            np.array([0x8b, 0x91, 0xad]),   # ad918b -> BGR: 139,145,173
            np.array([0x20, 0x28, 0x42]),   # 422820 -> BGR: 32,40,66 (очень темный)
        ]
        # Цвета призраков (сине-голубые оттенки)
        ghost_colors = [
            np.array([0xd4, 0xab, 0x62]),   # 62abd4 -> BGR: 212,171,98 (голубой)
            np.array([0xa1, 0x7e, 0x57]),   # 577ea1 -> BGR: 161,126,87 (темно-голубой)
            np.array([0xd0, 0xd5, 0xcd]),   # cdd5d0 -> BGR: 208,213,205 (серо-голубой)
            #np.array([0x1a, 0x13, 0x25]),   # 25131a -> BGR: 26,19,37 (очень темный)
        ]
        # Инициализируем переменные
        pacman_pos = (None, None)
        ghost_pos = (None, None)
        pg_pos = (None, None)
        # Преобразуем в HSV для лучшего разделения цветов
        hsv = cv2.cvtColor(game_field, cv2.COLOR_BGR2HSV)
        # --- УЛУЧШЕННЫЙ ПОИСК ПАКМАНА ---
        pacman_mask = np.zeros((game_field.shape[0], game_field.shape[1]), dtype=np.uint8)
        # Способ 1: Поиск по HSV диапазонам (лучше для телесных/коричневых оттенков)
        # Телесные/коричневые оттенки в HSV
        hsv_lower1 = np.array([0, 30, 100])    # Низкая насыщенность, средняя яркость
        hsv_upper1 = np.array([30, 150, 230])  # Красноватые/оранжевые оттенки
        # Сероватые/бежевые оттенки
        hsv_lower2 = np.array([0, 0, 150])     # Низкая насыщенность, высокая яркость
        hsv_upper2 = np.array([180, 50, 255])
        # Создаем маски по HSV
        hsv_mask1 = cv2.inRange(hsv, hsv_lower1, hsv_upper1)
        hsv_mask2 = cv2.inRange(hsv, hsv_lower2, hsv_upper2)
        pacman_hsv_mask = cv2.bitwise_or(hsv_mask1, hsv_mask2)
        # Способ 2: Поиск по BGR диапазонам (как было, но с улучшениями)
        for color in pacman_main_colors:
            # Широкий диапазон для основного цвета
            lower_bound = color - np.array([40, 40, 40])
            upper_bound = color + np.array([40, 40, 40])
            lower_bound = np.clip(lower_bound, 0, 255)
            upper_bound = np.clip(upper_bound, 0, 255)
            color_mask = cv2.inRange(game_field, lower_bound, upper_bound)
            pacman_mask = cv2.bitwise_or(pacman_mask, color_mask)
        for color in pacman_secondary_colors:
            # Уже диапазон для вторичных цветов
            lower_bound = color - np.array([30, 30, 30])
            upper_bound = color + np.array([30, 30, 30])
            lower_bound = np.clip(lower_bound, 0, 255)
            upper_bound = np.clip(upper_bound, 0, 255)
            color_mask = cv2.inRange(game_field, lower_bound, upper_bound)
            pacman_mask = cv2.bitwise_or(pacman_mask, color_mask)
        # Объединяем HSV и BGR маски
        pacman_mask = cv2.bitwise_or(pacman_mask, pacman_hsv_mask)
        if debug_img is not None:
            cv2.imshow('Pacman Mask Before', pacman_mask)
        # Морфологические операции для улучшения маски
        kernel = np.ones((5, 5), np.uint8)
        pacman_mask = cv2.morphologyEx(pacman_mask, cv2.MORPH_CLOSE, kernel)
        pacman_mask = cv2.morphologyEx(pacman_mask, cv2.MORPH_OPEN, kernel)
        # Удаляем слишком маленькие области (шум)
        contours, _ = cv2.findContours(pacman_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            # Фильтруем контуры по размеру
            filtered_contours = []
            for contour in contours:
                area = cv2.contourArea(contour)
                # Ожидаемый размер пакмана
                if 50 < area < 5000:  # Пакман должен быть среднего размера
                    filtered_contours.append(contour)
            if filtered_contours:
                # Берем самый большой подходящий контур
                largest_contour = max(filtered_contours, key=cv2.contourArea)
                # Более точное определение центра
                M = cv2.moments(largest_contour)
                if M['m00'] > 0:
                    pacman_x = int(M['m10'] / M['m00'])
                    pacman_y = int(M['m01'] / M['m00'])
                    # Преобразуем координаты обратно в полный скриншот
                    pacman_x_full = pacman_x + game_field_left
                    pacman_y_full = pacman_y + game_field_top
                    pacman_pos = (pacman_x_full, pacman_y_full)
                    # Дополнительная проверка: форма должна быть примерно круглой
                    perimeter = cv2.arcLength(largest_contour, True)
                    if perimeter > 0:
                        circularity = 4 * np.pi * M['m00'] / (perimeter * perimeter)
                        if circularity > 0.5:  # Достаточно круглая форма
                            pacman_pos = (pacman_x_full, pacman_y_full)
                            if debug_img is not None:
                                print(f"Pacman detected at {pacman_pos}, circularity: {circularity:.2f}") 
        # Если не нашли пакмана обычным способом, пробуем найти по движению/позиции
        if pacman_pos[0] is None:
            # Попробуем найти самую яркую область в центре экрана (где обычно пакман)
            center_x, center_y = game_field.shape[1] // 2, game_field.shape[0] // 2
            search_radius = 100
            # Ищем области с высоким значением в HSV V канале (яркость)
            v_channel = hsv[:,:,2]
            search_area = v_channel[
                max(0, center_y-search_radius):min(game_field.shape[0], center_y+search_radius),
                max(0, center_x-search_radius):min(game_field.shape[1], center_x+search_radius)
            ]
            if search_area.size > 0:
                max_val = np.max(search_area)
                if max_val > 150:  # Достаточно яркая область
                    max_pos = np.unravel_index(np.argmax(search_area), search_area.shape)
                    pacman_x = center_x - search_radius + max_pos[1] + game_field_left
                    pacman_y = center_y - search_radius + max_pos[0] + game_field_top
                    pacman_pos = (pacman_x, pacman_y)
        # --- УЛУЧШЕННЫЙ ПОИСК ПРИЗРАКОВ ---
        ghost_mask = np.zeros((game_field.shape[0], game_field.shape[1]), dtype=np.uint8)
        # Поиск призраков по HSV (сине-голубые оттенки)
        ghost_hsv_lower = np.array([90, 50, 50])   # Синие оттенки
        ghost_hsv_upper = np.array([130, 255, 255])
        ghost_hsv_mask = cv2.inRange(hsv, ghost_hsv_lower, ghost_hsv_upper)
        # Поиск по BGR диапазонам
        for color in ghost_colors:
            lower_bound = color - np.array([30, 30, 30])
            upper_bound = color + np.array([30, 30, 30])
            lower_bound = np.clip(lower_bound, 0, 255)
            upper_bound = np.clip(upper_bound, 0, 255)
            color_mask = cv2.inRange(game_field, lower_bound, upper_bound)
            ghost_mask = cv2.bitwise_or(ghost_mask, color_mask)
        # Объединяем маски
        ghost_mask = cv2.bitwise_or(ghost_mask, ghost_hsv_mask)
        # Убираем области, уже определенные как пакман
        if pacman_mask is not None:
            ghost_mask[pacman_mask > 0] = 0
        if debug_img is not None:
            cv2.imshow('Ghost Mask', ghost_mask)
        # Морфологические операции
        ghost_mask = cv2.morphologyEx(ghost_mask, cv2.MORPH_CLOSE, kernel)
        ghost_mask = cv2.morphologyEx(ghost_mask, cv2.MORPH_OPEN, kernel)
        # Находим контуры призраков
        ghost_contours, _ = cv2.findContours(ghost_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        # Находим ближайшего призрака к пакману
        nearest_ghost_pos = (None, None)
        if pacman_pos[0] is not None and ghost_contours:
            min_distance = float('inf')
            for contour in ghost_contours:
                area = cv2.contourArea(contour)
                # Призраки обычно среднего размера
                if 100 < area < 3000:
                    M = cv2.moments(contour)
                    if M['m00'] > 0:
                        ghost_x = int(M['m10'] / M['m00'])
                        ghost_y = int(M['m01'] / M['m00'])
                        # Преобразуем координаты
                        ghost_x_full = ghost_x + game_field_left
                        ghost_y_full = ghost_y + game_field_top
                        distance = np.sqrt((ghost_x_full - pacman_pos[0])**2 + (ghost_y_full - pacman_pos[1])**2)
                        if distance < min_distance:
                            min_distance = distance
                            nearest_ghost_pos = (ghost_x_full, ghost_y_full)
        ghost_pos = nearest_ghost_pos
        # --- ПОИСК ПАКГАМОВ ---
        # Пакгамы обычно БЕЛЫЕ/СВЕТЛЫЕ маленькие точки
        # ИЛИ ярко-зеленые (если изменили цвет на 10ff00)
        pg_mask = np.zeros((game_field.shape[0], game_field.shape[1]), dtype=np.uint8)
        # Способ 1: Поиск ярко-зеленых точек (10ff00)
        # 10ff00 в HSV: зеленый оттенок
        green_lower = np.array([50, 100, 100])    # Нижняя граница зеленого
        green_upper = np.array([90, 255, 255])    # Верхняя граница зеленого
        green_mask = cv2.inRange(hsv, green_lower, green_upper)
        # Способ 2: Поиск белых/светлых точек
        white_lower = np.array([0, 0, 200])
        white_upper = np.array([180, 50, 255])
        white_mask = cv2.inRange(hsv, white_lower, white_upper)
        # Объединяем маски (зеленые и белые)
        pg_mask = cv2.bitwise_or(green_mask, white_mask)
        # Убираем области пакмана и призраков
        if pacman_mask is not None:
            pg_mask[pacman_mask > 0] = 0
        if ghost_mask is not None:
            pg_mask[ghost_mask > 0] = 0
        # Находим маленькие круглые объекты (пакгамы)
        pg_contours, _ = cv2.findContours(pg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        nearest_pg_pos = (None, None)
        if pacman_pos[0] is not None and pg_contours:
            min_distance = float('inf')
            for contour in pg_contours:
                area = cv2.contourArea(contour)
                # Пакгамы маленькие
                if 5 < area < 100:
                    M = cv2.moments(contour)
                    if M['m00'] > 0:
                        pg_x = int(M['m10'] / M['m00'])
                        pg_y = int(M['m01'] / M['m00'])
                        # Преобразуем координаты
                        pg_x_full = pg_x + game_field_left
                        pg_y_full = pg_y + game_field_top
                        distance = np.sqrt((pg_x_full - pacman_pos[0])**2 + (pg_y_full - pacman_pos[1])**2)
                        if distance < min_distance:
                            min_distance = distance
                            nearest_pg_pos = (pg_x_full, pg_y_full)
        pg_pos = nearest_pg_pos
        # --- СОЗДАНИЕ СТЕЙТА И ОТЛАДКА ---
        current_state = 0
        allowed_moves_ = []
        if pacman_pos[0] is not None:
            # Если есть пакман, пытаемся определить доступные ходы
            wsz = 20
            # Простая проверка по яркости (используем игровое поле)
            gray = cv2.cvtColor(game_field, cv2.COLOR_BGR2GRAY)
            # Переводим координаты пакмана в систему координат игрового поля
            pacman_x_field = pacman_pos[0] - game_field_left
            pacman_y_field = pacman_pos[1] - game_field_top
            # Проверяем четыре направления
            directions = ['up', 'down', 'left', 'right']
            checks = [
                (pacman_y_field-wsz, pacman_y_field-5, pacman_x_field-5, pacman_x_field+5),  # up
                (pacman_y_field+5, pacman_y_field+wsz, pacman_x_field-5, pacman_x_field+5),  # down
                (pacman_y_field-5, pacman_y_field+5, pacman_x_field-wsz, pacman_x_field-5),  # left
                (pacman_y_field-5, pacman_y_field+5, pacman_x_field+5, pacman_x_field+wsz),  # right
            ]
            for direction, (y1, y2, x1, x2) in zip(directions, checks):
                if y1 >= 0 and y2 < gray.shape[0] and x1 >= 0 and x2 < gray.shape[1]:
                    area = gray[y1:y2, x1:x2]
                    if area.size > 0 and np.mean(area) > 100:  # Не слишком темно
                        allowed_moves_.append(direction)
            # Формируем состояние если все координаты есть
            if (ghost_pos[0] is not None and ghost_pos[1] is not None and
                pg_pos[0] is not None and pg_pos[1] is not None):
                try:
                    ghost_dir = xy_state(pacman_pos[0], pacman_pos[1], ghost_pos[0], ghost_pos[1])
                    pg_dir = xy_state(pacman_pos[0], pacman_pos[1], pg_pos[0], pg_pos[1])
                    num_moves = len(allowed_moves_)
                    if num_moves < 2:
                        num_moves = 2
                    elif num_moves > 4:
                        num_moves = 4
                    current_state = (ghost_dir * 8 + pg_dir) * 3 + (num_moves - 2)
                    if debug_img is not None:
                        print(f"CV State: {current_state}, Pacman: {pacman_pos}, "
                            f"Ghost: {ghost_pos}, PG: {pg_pos}, Moves: {allowed_moves_}")
                except Exception as e:
                    if debug_img is not None:
                        print(f"State calculation error: {e}")
        # Отладочная визуализация
        if debug_img is not None:
            # Показываем промежуточные маски
            cv2.imshow('Pacman Mask Final', pacman_mask)
            cv2.waitKey(1)
        # Возвращаем состояние и обнаруженные объекты
        detected_objects = {
            'pacman': pacman_pos,
            'ghost': ghost_pos,
            'pg': pg_pos,
            'allowed_moves': allowed_moves_
        }
        return current_state, detected_objects